# Mitra Regressor — DIMER E2E tabular regression tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/mitra-regressor-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/mitra-regressor-pipeline/blob/main/tutorials/mitra_regressor_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-autogluon%2Fmitra--regressor-ffcc4d?style=flat)](https://huggingface.co/autogluon/mitra-regressor) [![Upstream](https://img.shields.io/badge/Upstream-autogluon%2Fautogluon-181717?style=flat&logo=github&logoColor=white)](https://github.com/autogluon/autogluon) [![arXiv](https://img.shields.io/badge/arXiv-2508.02927-b31b1b.svg)](https://arxiv.org/abs/2508.02927)

**Profile:** `E2E`  
**Notebook specification:** DIMER Notebook Specification 1.1 — **standalone** (§3.6)  
**Capability:** end-to-end tabular regression with the pinned `autogluon/mitra-regressor` checkpoint through AutoGluon: verified model acquisition, validated support data, in-context evaluation against executable baselines, optional GPU fine-tuning, new-data inference, a deployable predictor bundle and its fresh-boundary reload

**This notebook is standalone.** It carries the repository's pipeline module (`mitra_pipeline/tutorial_api.py` at revision `d0fb1a81460d`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `5f277aa8f69042d39d6ac3612aed18bb9279bd95` (~303 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

Mitra is an in-context tabular foundation model served through AutoGluon's `TabularPredictor`: with `fine_tune=False`, `fit` registers the support rows and the model configuration and no weight is gradient-updated; fine-tuning is an opt-in gate that needs a GPU. The upstream project supplies the model and the checkpoint; the carried package adds the pinned snapshot scheme, the table validation, capping and overlap checks, the metric set, the `validate_inputs` / `training_mean_baseline` / `evaluation_report` helpers, and the archive-safety and artifact-manifest functions. The default sample is scikit-learn's bundled diabetes table; its metrics are tutorial sanity evidence, not a benchmark or production claim.

**Learning objectives:** install the pinned runtime, read what the carried package guarantees, resolve and digest-verify the immutable upstream checkpoint and stage it as an offline Hugging Face snapshot, load a public sample or your own CSV(s) and validate them into an input manifest, evaluate the pretrained model on a holdout and an independent test partition against the training-mean, median, LightGBM and Random Forest baselines, optionally fine-tune on a GPU with holdout-based selection, write an evaluation report, optionally score new rows, export the deployable AutoGluon predictor bundle with its manifest and provenance, and prove it reloads from a fresh directory.

**This notebook does not demonstrate:** classification, forecasting, calibrated per-prediction uncertainty intervals, or any deployment tolerance band. Predictions are **continuous point estimates only**; the fine-tuning path runs only on a GPU and only when `RUN_FINE_TUNING` is switched on.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.10–3.13 as required by AutoGluon 1.5.0). The default path runs on CPU and uses CUDA automatically when available; the fine-tuning gate requires a GPU. The pinned `autogluon.tabular[mitra]==1.5.0` install (with its torch) is the largest download of the run.
- **Knowledge:** basic pandas; what a holdout, an independent test partition, MAE, RMSE and R² are.
- **Data:** the default sample is scikit-learn's bundled diabetes table (442 rows, 10 numeric features), loaded from the installed package, so nothing is downloaded and no private data is needed. BYOD (one labelled CSV, or pre-split `train.csv`/`val.csv`/`test.csv` — the repository's `examples/sample-data/` archives can be supplied this way) is selected through `DATA_SOURCE` and is off by default so the sample path runs top-to-bottom without interaction. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `autogluon/mitra-regressor` snapshot (~303 MB in total) at revision `5f277aa8f690…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `numpy`, `pandas`, `sklearn` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'autogluon.tabular[mitra]==1.5.0',
    'lightgbm==4.6.0',
    'huggingface-hub==0.36.2',
]
NOTEBOOK_SOURCE = {
    'repository': 'mitra-regressor-pipeline',
    'repository_revision': 'd0fb1a81460d094c82bd30238d01db1e29a3296a',
    'embedded_module': 'mitra_pipeline/tutorial_api.py',
    'embedded_modules': ['mitra_pipeline/tutorial_api.py'],
    'module_sha256': 'a6b564ceb6e5d45363f9d4e3a66bb662e951bada7638ddc33d5fb1155e24a569',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '1.1',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, numpy, pandas, sklearn
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'numpy': numpy.__version__, 'pandas': pandas.__version__, 'sklearn': sklearn.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `mitra_pipeline/` @ `d0fb1a81460d`)

The next 1 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/1:** `mitra_pipeline/tutorial_api.py`

In [ ]:
"""Release-grade public API for the Mitra Regressor notebook surface.

This module owns the repository-facing behavior the notebooks must exercise directly:
regression data validation, Mitra fit/predict calls, model snapshot locking, and predictor
artifact validation. Imports that pull in heavy ML dependencies are intentionally delayed so
archive and manifest checks remain unit-testable in a lightweight CI environment.
"""

from __future__ import annotations

import csv
import hashlib
import io
import json
import os
import random
import shutil
import stat
import tempfile
import urllib.request
import zipfile
from collections.abc import Callable, Iterable, Mapping, Sequence
from pathlib import Path, PurePosixPath
from typing import Any

import numpy as np
import pandas as pd

# Fleet snapshot identity (DIMER Notebook Specification 1.1, ST3/MOD13). The pinned upstream model is unchanged;
# these are the fleet-standard names for the same repository, revision, license and snapshot key. The published
# PINNED_REVISION spelling stays as an alias of MODEL_REVISION.
MODEL_ID = "autogluon/mitra-regressor"
MODEL_REVISION = "5f277aa8f69042d39d6ac3612aed18bb9279bd95"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "mitra-regressor"
MANIFEST_NAME = "dimer-base-manifest.json"
# Root-level package: the repository root is one level up from this file.
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
WEIGHTS_FILE = "model.safetensors"
CONFIG_FILE = "config.json"

PINNED_REVISION = MODEL_REVISION
WEIGHTS_SHA256 = "d8e75c62af0bec2fd404b0ad20a442d951d43ca6d331315cfcc0509b54f2c642"
CONFIG_SHA256 = "2bc1ed5047f7c25368245e8ad32540a5fa28940b1ec05d3f1f454a09ff5384c1"
METRIC_IDS = ("mae", "rmse", "r2")  # the ids `regression_metrics` reports

MAX_TRAIN_ROWS = 10_000
MAX_FEATURES = 500
MIN_TRAIN_ROWS = 50

ARTIFACT_FORMAT = "dimer-autogluon-predictor"
ARTIFACT_FORMAT_VERSION = 1
ARTIFACT_MANIFEST = "artifact_manifest.json"
RUN_METADATA = "tutorial_run_metadata.json"
DIMER_MODEL_MANIFEST = "dimer-model-manifest.json"

MAX_ARCHIVE_MEMBER_BYTES = 1 * 1024**3
MAX_ARCHIVE_EXPANDED_BYTES = 4 * 1024**3
MAX_COMPRESSION_RATIO = 200.0

REGRESSION_LOWER_IS_BETTER = {
    "root_mean_squared_error",
    "mean_squared_error",
    "mean_absolute_error",
    "median_absolute_error",
    "mean_absolute_percentage_error",
    "symmetric_mean_absolute_percentage_error",
    "root_mean_squared_logarithmic_error",
}
MITRA_METRIC_MAP = {
    "mean_absolute_error": "mae",
    "root_mean_squared_error": "rmse",
}

REQUIRED_RUN_METADATA_FIELDS = {
    "artifact_format",
    "artifact_format_version",
    "base_model",
    "base_model_revision",
    "weights_sha256",
    "config_sha256",
    "autogluon_version",
    "python_version",
    "problem_type",
    "target_column",
    "features",
    "mode",
    "selection_basis",
}


def sha256_file(path: str | Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def _assert_hex_digest(value: str, label: str) -> str:
    digest = str(value).strip().lower()
    if len(digest) != 64 or any(ch not in "0123456789abcdef" for ch in digest):
        raise ValueError(f"{label} must be a 64-character hexadecimal SHA-256 digest.")
    return digest


def read_csv_bytes(payload: bytes, label: str) -> pd.DataFrame:
    """Parse UTF-8/BOM CSV bytes while rejecting duplicate raw headers before pandas renames them."""
    try:
        text = payload.decode("utf-8-sig")
    except UnicodeDecodeError as exc:
        raise ValueError(f"{label}: expected UTF-8 CSV input.") from exc

    rows = csv.reader(io.StringIO(text, newline=""))
    header = next((row for row in rows if row and not (len(row) == 1 and not row[0].strip())), [])
    if not header:
        raise ValueError(f"{label}: CSV has no header row.")

    seen: set[str] = set()
    duplicates: list[str] = []
    for name in header:
        if name in seen and name not in duplicates:
            duplicates.append(name)
        seen.add(name)
    if duplicates:
        raise ValueError(f"{label}: duplicate column names are not supported: {duplicates}")

    return pd.read_csv(io.BytesIO(payload))


def validate_labeled_frame(
    frame: pd.DataFrame,
    target_column: str,
    *,
    name: str,
    drop_columns: Iterable[str] = (),
    min_rows: int = 2,
    require_variation: bool = False,
) -> tuple[pd.DataFrame, list[str], dict[str, Any]]:
    """Validate one labelled regression table using the repository's user-facing contract."""
    if frame.columns.duplicated().any():
        duplicates = list(frame.columns[frame.columns.duplicated()])
        raise ValueError(f"{name}: duplicate column names are not supported: {duplicates}")
    if target_column not in frame.columns:
        raise ValueError(f"{name}: target {target_column!r} not found.")

    drops = [c for c in drop_columns if c != target_column and c in frame.columns]
    out = frame.drop(columns=drops, errors="ignore").copy()
    raw_target = out[target_column]
    numeric_target = pd.to_numeric(raw_target, errors="coerce")
    non_numeric = raw_target.notna() & numeric_target.isna()
    if bool(non_numeric.any()):
        examples = raw_target[non_numeric].astype(str).head(5).tolist()
        raise ValueError(f"{name}: target must be numeric; invalid examples: {examples}")
    out[target_column] = numeric_target

    rows_before = len(out)
    out = out.dropna(subset=[target_column]).copy()
    dropped_target_rows = rows_before - len(out)
    target_values = out[target_column].to_numpy(dtype=float)
    if not np.isfinite(target_values).all():
        raise ValueError(f"{name}: target contains infinite values; use finite numeric regression targets only.")

    features = [c for c in out.columns if c != target_column]
    errors: list[str] = []
    if len(out) < min_rows:
        errors.append(f"use at least {min_rows} labelled rows after missing-target drops")
    if not features:
        errors.append("no feature columns remain")
    if len(features) > MAX_FEATURES:
        errors.append(f"{len(features)} features exceed the {MAX_FEATURES}-feature Mitra ceiling")
    if require_variation and out[target_column].nunique(dropna=True) < 2:
        errors.append("training target has no variation")
    if errors:
        raise ValueError(f"{name} is not ready: " + "; ".join(errors))

    report = {
        "rows_before_target_drop": int(rows_before),
        "rows_after_target_drop": int(len(out)),
        "dropped_missing_target_rows": int(dropped_target_rows),
        "exact_duplicate_rows": int(out.duplicated().sum()),
        "feature_count": int(len(features)),
    }
    return out, features, report


def cap_training_rows(
    frame: pd.DataFrame,
    target_column: str,
    *,
    seed: int,
    max_rows: int = MAX_TRAIN_ROWS,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    if max_rows > MAX_TRAIN_ROWS:
        raise ValueError(f"max_rows cannot exceed Mitra's {MAX_TRAIN_ROWS:,}-row ceiling.")
    before = len(frame)
    if before <= max_rows:
        return frame.copy(), {"applied": False, "before": before, "after": before}
    capped = frame.sample(n=max_rows, random_state=seed).copy()
    if capped[target_column].nunique(dropna=True) < 2:
        raise ValueError("Capped training split has no target variation; provide a representative pre-split set.")
    return capped, {"applied": True, "before": before, "after": len(capped)}


def split_overlap_report(named_frames: dict[str, pd.DataFrame]) -> dict[str, int]:
    """Detect exact record overlap across labelled partitions without mutating them."""
    names = list(named_frames)
    hashes = {
        name: set(pd.util.hash_pandas_object(frame, index=False).astype("uint64").tolist())
        for name, frame in named_frames.items()
    }
    report: dict[str, int] = {}
    for i, left in enumerate(names):
        for right in names[i + 1 :]:
            report[f"{left}_vs_{right}"] = len(hashes[left].intersection(hashes[right]))
    return report


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch

        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except Exception:
        pass


def fit_mitra_predictor(
    train_data: pd.DataFrame,
    *,
    target_column: str,
    eval_metric: str,
    path: str | Path,
    fine_tune: bool,
    time_limit: int,
    seed: int,
    fine_tune_steps: int | None = None,
    max_memory_usage_ratio: float = 1.10,
):
    """Fit/register Mitra through the repository's supported notebook API.

    With ``fine_tune=False`` AutoGluon's ``fit`` registers support/context rows and model
    configuration; Mitra weights are not gradient-updated. With ``fine_tune=True`` the Mitra
    weights are adapted for the requested number of steps, subject to the time limit.
    """
    if eval_metric not in MITRA_METRIC_MAP:
        raise ValueError(f"Unsupported eval_metric {eval_metric!r}; choose {sorted(MITRA_METRIC_MAP)}")
    if fine_tune and (fine_tune_steps is None or fine_tune_steps <= 0):
        raise ValueError("fine_tune_steps must be a positive integer when fine_tune=True.")

    seed_everything(seed)
    hp: dict[str, Any] = {
        "fine_tune": bool(fine_tune),
        "seed": int(seed),
        "metric": MITRA_METRIC_MAP[eval_metric],
    }
    if fine_tune:
        hp["fine_tune_steps"] = int(fine_tune_steps)

    from autogluon.tabular import TabularPredictor

    predictor = TabularPredictor(
        label=target_column,
        problem_type="regression",
        eval_metric=eval_metric,
        path=str(path),
        verbosity=2,
    )
    predictor.fit(
        train_data,
        hyperparameters={"MITRA": hp},
        fit_weighted_ensemble=False,
        time_limit=int(time_limit),
        ag_args_fit={"max_memory_usage_ratio": float(max_memory_usage_ratio)},
    )
    trained = list(predictor.model_names())
    if not trained or not any("mitra" in model.lower() for model in trained):
        raise RuntimeError(f"Expected Mitra to fit/register; AutoGluon returned models={trained}.")
    return predictor


def normalize_autogluon_regression_metrics(raw: dict[str, Any]) -> dict[str, float]:
    return {
        key: float(-value if key in REGRESSION_LOWER_IS_BETTER else value)
        for key, value in raw.items()
    }


def regression_metrics(y_true: Iterable[float], y_pred: Iterable[float]) -> dict[str, float]:
    truth = np.asarray(list(y_true), dtype=float)
    pred = np.asarray(list(y_pred), dtype=float)
    if truth.shape != pred.shape:
        raise ValueError(f"y_true shape {truth.shape} != y_pred shape {pred.shape}")
    err = truth - pred
    mae = float(np.mean(np.abs(err)))
    rmse = float(np.sqrt(np.mean(err**2)))
    denom = float(np.sum((truth - np.mean(truth)) ** 2))
    r2 = float(1.0 - np.sum(err**2) / denom) if denom > 0 else float("nan")
    return {"mae": mae, "rmse": rmse, "r2": r2}


def validate_inference_frame(
    frame: pd.DataFrame,
    required_features: Iterable[str],
    *,
    output_column: str = "prediction",
) -> tuple[pd.DataFrame, list[str]]:
    if frame.columns.duplicated().any():
        duplicates = list(frame.columns[frame.columns.duplicated()])
        raise ValueError(f"Inference CSV contains duplicate column names: {duplicates}")
    required = list(required_features)
    missing = [column for column in required if column not in frame.columns]
    if missing:
        raise ValueError(f"Inference CSV is missing required feature columns: {missing}")
    if output_column in frame.columns:
        raise ValueError(f"Inference CSV already contains {output_column!r}; rename or remove it first.")
    extra = [column for column in frame.columns if column not in required]
    return frame.reindex(columns=required).copy(), extra


def predict_regression(predictor, frame: pd.DataFrame, required_features: Iterable[str]) -> np.ndarray:
    features = list(required_features)
    missing = [column for column in features if column not in frame.columns]
    if missing:
        raise ValueError(f"Prediction frame is missing required features: {missing}")
    predictions = predictor.predict(frame.reindex(columns=features))
    return np.asarray(predictions, dtype=float)


def _download(url: str, destination: Path, *, timeout: int = 30) -> None:
    with urllib.request.urlopen(url, timeout=timeout) as response, open(destination, "wb") as handle:
        shutil.copyfileobj(response, handle)


def stage_verified_hf_snapshot(
    weights_path: str | Path,
    config_path: str | Path,
    *,
    hf_home: str | Path,
) -> Path:
    """Verify exact Mitra bytes and stage them as the immutable pinned Hugging Face snapshot."""
    weights = Path(weights_path)
    config = Path(config_path)
    if sha256_file(weights) != WEIGHTS_SHA256:
        raise RuntimeError("model.safetensors checksum mismatch for the pinned Mitra Regressor release.")
    if sha256_file(config) != CONFIG_SHA256:
        raise RuntimeError("config.json checksum mismatch for the pinned Mitra Regressor release.")

    home = Path(hf_home)
    repo = home / "hub" / ("models--" + MODEL_ID.replace("/", "--"))
    snapshot = repo / "snapshots" / PINNED_REVISION
    refs = repo / "refs"
    snapshot.mkdir(parents=True, exist_ok=True)
    refs.mkdir(parents=True, exist_ok=True)
    shutil.copy2(weights, snapshot / "model.safetensors")
    shutil.copy2(config, snapshot / "config.json")
    (refs / "main").write_text(PINNED_REVISION, encoding="utf-8")

    os.environ["HF_HOME"] = str(home)
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"

    from huggingface_hub import hf_hub_download

    for filename, expected_digest in (
        ("model.safetensors", WEIGHTS_SHA256),
        ("config.json", CONFIG_SHA256),
    ):
        resolved = Path(
            hf_hub_download(
                repo_id=MODEL_ID,
                filename=filename,
                revision=PINNED_REVISION,
                local_files_only=True,
            )
        ).resolve()
        expected = (snapshot / filename).resolve()
        if resolved != expected:
            raise RuntimeError(f"Offline resolver mismatch for {filename}: {resolved} != {expected}")
        if sha256_file(resolved) != expected_digest:
            raise RuntimeError(f"Resolved {filename} digest changed after staging.")
    return snapshot


def _validate_zip_members(zf: zipfile.ZipFile, destination: Path) -> list[zipfile.ZipInfo]:
    destination = destination.resolve()
    total = 0
    files: list[zipfile.ZipInfo] = []
    seen_paths: set[str] = set()
    seen_parent_paths: set[str] = set()
    for info in zf.infolist():
        name = info.filename
        if not name or info.is_dir():
            continue
        if "\\" in name:
            raise RuntimeError(f"Backslash archive paths are not allowed: {name!r}")
        member = PurePosixPath(name)
        if member.is_absolute() or ".." in member.parts:
            raise RuntimeError(f"Unsafe archive member path: {name!r}")
        normalized_name = member.as_posix()
        if normalized_name in seen_paths:
            raise RuntimeError(f"Duplicate archive member path is not allowed: {name!r}")
        parent_paths = {PurePosixPath(*member.parts[:i]).as_posix() for i in range(1, len(member.parts))}
        if normalized_name in seen_parent_paths or parent_paths.intersection(seen_paths):
            raise RuntimeError(f"Archive member path conflicts with a file/directory boundary: {name!r}")
        seen_paths.add(normalized_name)
        seen_parent_paths.update(parent_paths)
        mode = (info.external_attr >> 16) & 0o170000
        if mode == stat.S_IFLNK:
            raise RuntimeError(f"Symlink entries are not allowed: {name!r}")
        if info.file_size > MAX_ARCHIVE_MEMBER_BYTES:
            raise RuntimeError(f"Archive member is too large: {name!r} ({info.file_size:,} bytes).")
        if info.compress_size > 0 and info.file_size / info.compress_size > MAX_COMPRESSION_RATIO:
            raise RuntimeError(f"Archive member has suspicious compression ratio: {name!r}.")
        total += info.file_size
        if total > MAX_ARCHIVE_EXPANDED_BYTES:
            raise RuntimeError("Archive expanded size exceeds the configured safety ceiling.")
        target = (destination / Path(*member.parts)).resolve()
        if target != destination and destination not in target.parents:
            raise RuntimeError(f"Archive member escapes extraction root: {name!r}")
        files.append(info)
    return files


def safe_extract_archive(zip_path: str | Path, destination: str | Path) -> Path:
    destination_path = Path(destination)
    with zipfile.ZipFile(zip_path) as zf:
        # Validate the complete archive before touching any prior extraction destination.
        infos = _validate_zip_members(zf, destination_path)
        if destination_path.is_symlink():
            raise RuntimeError("Archive extraction destination must not be a symlink.")
        destination_path.parent.mkdir(parents=True, exist_ok=True)
        staging = Path(tempfile.mkdtemp(prefix=f".{destination_path.name}.extract-", dir=destination_path.parent))
        try:
            for info in infos:
                member = PurePosixPath(info.filename)
                target = staging.joinpath(*member.parts)
                target.parent.mkdir(parents=True, exist_ok=True)
                with zf.open(info) as source, open(target, "wb") as sink:
                    shutil.copyfileobj(source, sink)
            if destination_path.exists():
                shutil.rmtree(destination_path)
            staging.replace(destination_path)
        except Exception:
            shutil.rmtree(staging, ignore_errors=True)
            raise
    return destination_path


def validate_dimer_model_package(
    zip_path: str | Path,
    destination: str | Path,
    *,
    expected_archive_sha256: str | None = None,
) -> tuple[Path, Path, dict[str, Any]]:
    """Validate the normative DIMER offline Mitra package before any model load."""
    zip_path = Path(zip_path)
    if expected_archive_sha256:
        expected = _assert_hex_digest(expected_archive_sha256, "expected_archive_sha256")
        actual = sha256_file(zip_path)
        if actual != expected:
            raise RuntimeError(f"DIMER model ZIP checksum mismatch: expected {expected}; got {actual}.")

    root = safe_extract_archive(zip_path, destination)
    manifest_path = root / DIMER_MODEL_MANIFEST
    if not manifest_path.is_file():
        raise RuntimeError(
            f"DIMER model ZIP must contain root-level {DIMER_MODEL_MANIFEST}; legacy weight-only ZIPs do not satisfy Notebook Spec 1.0."
        )
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("schema_version") != 1:
        raise RuntimeError("Unsupported DIMER model manifest schema_version.")
    if manifest.get("model_id") != MODEL_ID:
        raise RuntimeError(f"DIMER package model_id must be {MODEL_ID!r}.")
    if manifest.get("revision") != PINNED_REVISION:
        raise RuntimeError(f"DIMER package revision must be {PINNED_REVISION!r}.")

    declared = manifest.get("files")
    if not isinstance(declared, list) or not declared:
        raise RuntimeError("DIMER model manifest must contain a non-empty files list.")
    entries: dict[str, dict[str, Any]] = {}
    for entry in declared:
        if not isinstance(entry, dict):
            raise RuntimeError("DIMER model manifest file entries must be objects.")
        path = str(entry.get("path", ""))
        if not path or "\\" in path:
            raise RuntimeError(f"Invalid DIMER manifest path: {path!r}")
        member = PurePosixPath(path)
        if member.is_absolute() or ".." in member.parts or path in entries:
            raise RuntimeError(f"Unsafe or duplicate DIMER manifest path: {path!r}")
        entries[path] = entry

    for required in ("model.safetensors", "config.json"):
        if required not in entries:
            raise RuntimeError(f"DIMER model manifest is missing required file {required!r}.")

    actual_files = {
        path.relative_to(root).as_posix()
        for path in root.rglob("*")
        if path.is_file() and path != root / DIMER_MODEL_MANIFEST
    }
    if actual_files != set(entries):
        missing = sorted(set(entries) - actual_files)
        unexpected = sorted(actual_files - set(entries))
        raise RuntimeError(f"DIMER model package file inventory mismatch; missing={missing}, unexpected={unexpected}")

    for rel, entry in entries.items():
        path = root / rel
        expected_size = int(entry.get("size", -1))
        expected_digest = _assert_hex_digest(str(entry.get("sha256", "")), f"sha256 for {rel}")
        if path.stat().st_size != expected_size:
            raise RuntimeError(f"DIMER model package size mismatch for {rel}.")
        if sha256_file(path) != expected_digest:
            raise RuntimeError(f"DIMER model package digest mismatch for {rel}.")

    weights = root / "model.safetensors"
    config = root / "config.json"
    if sha256_file(weights) != WEIGHTS_SHA256 or sha256_file(config) != CONFIG_SHA256:
        raise RuntimeError("DIMER package bytes do not match the pinned Mitra Regressor release.")
    return weights, config, manifest


def _artifact_inventory(root: Path) -> list[dict[str, Any]]:
    inventory = []
    for path in sorted(p for p in root.rglob("*") if p.is_file() and p != root / ARTIFACT_MANIFEST):
        rel = path.relative_to(root).as_posix()
        if "\\" in rel or PurePosixPath(rel).is_absolute() or ".." in PurePosixPath(rel).parts:
            raise RuntimeError(f"Unsafe artifact path: {rel!r}")
        inventory.append({"path": rel, "size": path.stat().st_size, "sha256": sha256_file(path)})
    return inventory


def write_artifact_manifest(root: str | Path) -> Path:
    root = Path(root)
    metadata_path = root / RUN_METADATA
    if not metadata_path.is_file():
        raise RuntimeError(f"Cannot manifest artifact without {RUN_METADATA}.")
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    missing = sorted(REQUIRED_RUN_METADATA_FIELDS - set(metadata))
    if missing:
        raise RuntimeError(f"Run metadata missing required fields: {missing}")
    if metadata.get("artifact_format") != ARTIFACT_FORMAT:
        raise RuntimeError("Run metadata artifact_format mismatch.")
    if metadata.get("artifact_format_version") != ARTIFACT_FORMAT_VERSION:
        raise RuntimeError("Run metadata artifact_format_version mismatch.")

    manifest = {
        "schema_version": 1,
        "artifact_format": ARTIFACT_FORMAT,
        "artifact_format_version": ARTIFACT_FORMAT_VERSION,
        "base_model": metadata["base_model"],
        "base_model_revision": metadata["base_model_revision"],
        "problem_type": metadata["problem_type"],
        "metadata_file": RUN_METADATA,
        "files": _artifact_inventory(root),
    }
    path = root / ARTIFACT_MANIFEST
    path.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8")
    return path


def validate_artifact_directory(
    root: str | Path,
    *,
    expected_model_id: str = MODEL_ID,
    expected_revision: str = PINNED_REVISION,
) -> tuple[dict[str, Any], dict[str, Any]]:
    """Verify format, provenance, inventory, sizes and digests before deserializing predictor state."""
    root = Path(root)
    manifest_path = root / ARTIFACT_MANIFEST
    metadata_path = root / RUN_METADATA
    if not manifest_path.is_file():
        raise RuntimeError(f"Predictor artifact is missing required {ARTIFACT_MANIFEST}.")
    if not metadata_path.is_file():
        raise RuntimeError(f"Predictor artifact is missing required {RUN_METADATA}.")

    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    if manifest.get("schema_version") != 1:
        raise RuntimeError("Unsupported artifact manifest schema_version.")
    if manifest.get("metadata_file") != RUN_METADATA:
        raise RuntimeError(f"Artifact manifest metadata_file must be {RUN_METADATA!r}.")
    if manifest.get("artifact_format") != ARTIFACT_FORMAT or metadata.get("artifact_format") != ARTIFACT_FORMAT:
        raise RuntimeError("Predictor artifact_format is not the DIMER AutoGluon predictor format.")
    if (
        manifest.get("artifact_format_version") != ARTIFACT_FORMAT_VERSION
        or metadata.get("artifact_format_version") != ARTIFACT_FORMAT_VERSION
    ):
        raise RuntimeError("Predictor artifact format version is incompatible with this notebook.")

    missing = sorted(REQUIRED_RUN_METADATA_FIELDS - set(metadata))
    if missing:
        raise RuntimeError(f"Predictor provenance is missing required fields: {missing}")
    if metadata.get("base_model") != expected_model_id or manifest.get("base_model") != expected_model_id:
        raise RuntimeError(f"Predictor base model must be {expected_model_id!r}.")
    if metadata.get("base_model_revision") != expected_revision or manifest.get("base_model_revision") != expected_revision:
        raise RuntimeError(f"Predictor base model revision must be {expected_revision!r}.")
    if metadata.get("problem_type") != "regression" or manifest.get("problem_type") != "regression":
        raise RuntimeError("Predictor artifact is not a regression artifact.")
    if metadata.get("weights_sha256") != WEIGHTS_SHA256 or metadata.get("config_sha256") != CONFIG_SHA256:
        raise RuntimeError("Predictor provenance does not identify the pinned Mitra Regressor bytes.")

    declared = manifest.get("files")
    if not isinstance(declared, list) or not declared:
        raise RuntimeError("Artifact manifest contains no file inventory.")
    entries: dict[str, dict[str, Any]] = {}
    for entry in declared:
        if not isinstance(entry, dict):
            raise RuntimeError("Artifact manifest file entries must be objects.")
        rel = str(entry.get("path", ""))
        member = PurePosixPath(rel)
        if not rel or "\\" in rel or member.is_absolute() or ".." in member.parts or rel in entries:
            raise RuntimeError(f"Unsafe or duplicate artifact manifest path: {rel!r}")
        entries[rel] = entry

    actual = {
        path.relative_to(root).as_posix()
        for path in root.rglob("*")
        if path.is_file() and path != root / ARTIFACT_MANIFEST
    }
    if actual != set(entries):
        missing_files = sorted(set(entries) - actual)
        unexpected_files = sorted(actual - set(entries))
        raise RuntimeError(
            f"Artifact inventory mismatch; missing={missing_files}, unexpected={unexpected_files}"
        )

    for rel, entry in entries.items():
        path = root / rel
        expected_size = int(entry.get("size", -1))
        expected_digest = _assert_hex_digest(str(entry.get("sha256", "")), f"sha256 for {rel}")
        if path.stat().st_size != expected_size:
            raise RuntimeError(f"Artifact file size mismatch: {rel}")
        if sha256_file(path) != expected_digest:
            raise RuntimeError(f"Artifact file digest mismatch: {rel}")

    if not (root / "predictor.pkl").is_file():
        raise RuntimeError("Artifact does not contain the required root-level AutoGluon predictor.pkl.")
    return manifest, metadata


# ---------------------------------------------------------------------------
# Fleet snapshot scheme (NOTEBOOK_SPEC 1.1 ST3/ST4, MOD13): manifest-driven verification and staging. The existing
# `stage_verified_hf_snapshot` (digest check + offline HF cache staging) stays the loader path and is called by
# `MitraRegressionPipeline.from_pretrained` after the manifest has been verified.
# ---------------------------------------------------------------------------


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local pinned snapshot against its manifest; raise naming the first mismatch.

    The manifest entries for ``model.safetensors`` and ``config.json`` must equal the package's own
    ``WEIGHTS_SHA256`` / ``CONFIG_SHA256`` constants, so the two can never diverge silently.
    """
    root = Path(path or DEFAULT_WEIGHTS_DIR)
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as handle:
        manifest = json.load(handle)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    entries = manifest.get("files", [])
    declared = {entry["path"]: entry["sha256"] for entry in entries}
    for filename, expected in ((WEIGHTS_FILE, WEIGHTS_SHA256), (CONFIG_FILE, CONFIG_SHA256)):
        if declared.get(filename) != expected:
            raise ValueError(
                f"manifest {filename} sha256 {declared.get(filename)!r} != package constant {expected!r}"
            )
    for entry in entries:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = sha256_file(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {"path": str(root), **manifest}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(repo_id=MODEL_ID, filename=relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a clone commits the manifest but git-ignores the
    weights). Returns the relative paths fetched; ``verify_snapshot`` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as handle:
        manifest = json.load(handle)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


class MitraRegressionPipeline:
    """Serving wrapper: the digest-verified pinned snapshot behind AutoGluon's Mitra regressor.

    ``from_pretrained`` stages and verifies the snapshot, then stages the verified bytes as the immutable offline
    Hugging Face snapshot AutoGluon resolves (``stage_verified_hf_snapshot``: HF_HUB_OFFLINE, no network path).
    ``fit`` registers the support rows through ``fit_mitra_predictor`` (in-context; gradient fine-tuning only with
    ``fine_tune=True``), ``evaluate`` and ``predict`` go through the repository's AutoGluon helpers.
    """

    def __init__(
        self,
        *,
        weights_path: Path,
        config_path: Path,
        snapshot_path: Path,
        device: str,
        source: str = "local-snapshot",
        predictor: Any = None,
        features: Sequence[str] | None = None,
    ) -> None:
        self.model_weight_path = Path(weights_path)
        self.config_path = Path(config_path)
        self.snapshot_path = Path(snapshot_path)
        self.device = device
        self.source = source
        self.predictor = predictor
        self.features: list[str] = list(features or [])
        self.target_column: str | None = None

    @classmethod
    def from_pretrained(
        cls,
        weights_dir: str | Path | None = None,
        *,
        allow_download: bool = False,
        hf_home: str | Path | None = None,
        device: str | None = None,
    ) -> MitraRegressionPipeline:
        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        stage_missing_files(root, allow_download=allow_download)
        verify_snapshot(root)
        home = Path(hf_home) if hf_home is not None else root / ".cache" / "hf"
        snapshot = stage_verified_hf_snapshot(root / WEIGHTS_FILE, root / CONFIG_FILE, hf_home=home)
        if device is None:
            try:
                import torch

                device = "cuda" if torch.cuda.is_available() else "cpu"
            except ImportError:
                device = "cpu"
        return cls(
            weights_path=root / WEIGHTS_FILE, config_path=root / CONFIG_FILE, snapshot_path=snapshot, device=device
        )

    def fit(
        self,
        train_data: pd.DataFrame,
        *,
        target_column: str,
        eval_metric: str,
        path: str | Path,
        fine_tune: bool = False,
        time_limit: int = 300,
        seed: int = 42,
        fine_tune_steps: int | None = None,
        max_memory_usage_ratio: float = 1.10,
    ) -> MitraRegressionPipeline:
        self.predictor = fit_mitra_predictor(
            train_data,
            target_column=target_column,
            eval_metric=eval_metric,
            path=path,
            fine_tune=fine_tune,
            time_limit=time_limit,
            seed=seed,
            fine_tune_steps=fine_tune_steps,
            max_memory_usage_ratio=max_memory_usage_ratio,
        )
        self.target_column = target_column
        self.features = [column for column in train_data.columns if column != target_column]
        self.source = "fine-tuned" if fine_tune else self.source
        return self

    def evaluate(self, frame: pd.DataFrame) -> dict[str, float]:
        """AutoGluon's evaluation of a labelled frame, normalised to positive error values."""
        if self.predictor is None:
            raise RuntimeError("Pipeline is not fitted; call fit(...) first")
        raw = self.predictor.evaluate(frame, auxiliary_metrics=True, silent=True)
        return normalize_autogluon_regression_metrics(raw)

    def predict(self, frame: pd.DataFrame) -> np.ndarray:
        if self.predictor is None:
            raise RuntimeError("Pipeline is not fitted; call fit(...) first")
        return predict_regression(self.predictor, frame, self.features)


# ---------------------------------------------------------------------------
# Role stages (DAT24 / EVAL21) on top of the existing validation and metric helpers.
# ---------------------------------------------------------------------------

INPUT_SCHEMA: dict[str, Any] = {
    "input": "pandas.DataFrame, one row per example; feature columns of any dtype plus a numeric target",
    "columns": "unique names; `drop_columns` are removed before validation",
    "target": (
        "coerced to numeric (non-numeric values are rejected); rows with a missing target are dropped and "
        "counted; infinite values are rejected; the training target must vary"
    ),
    "train_rows": [MIN_TRAIN_ROWS, MAX_TRAIN_ROWS],
    "eval_rows": [2, None],
    "features": [1, MAX_FEATURES],
    "inference_input": "every fitted feature column present; no `prediction` column; extras pass through",
    "preprocessing": (
        "none by the package (AutoGluon's Mitra handles raw columns); training rows above MAX_TRAIN_ROWS are "
        "capped by seeded sampling and the cap is reported"
    ),
}


def training_mean_baseline(
    train_targets: Iterable[float], holdout_targets: Iterable[float]
) -> dict[str, float]:
    """The trivial baseline: always predict the training mean (mae, rmse, r2: the `regression_metrics` ids)."""
    train = np.asarray(list(train_targets), dtype=float)
    holdout = np.asarray(list(holdout_targets), dtype=float)
    if train.size == 0 or holdout.size == 0 or not (np.isfinite(train).all() and np.isfinite(holdout).all()):
        raise ValueError("targets must be non-empty and finite")
    return regression_metrics(holdout, np.full(holdout.shape, float(train.mean())))


def validate_inputs(
    frame: pd.DataFrame,
    target_column: str | None = "target",
    *,
    drop_columns: Iterable[str] = (),
    min_rows: int = MIN_TRAIN_ROWS,
    require_variation: bool = True,
    feature_columns: Iterable[str] | None = None,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observed table properties, verdict).

    With a ``target_column`` the table is checked exactly as ``validate_labeled_frame`` checks it (its report:
    dropped missing-target rows, exact duplicates, feature count — is carried, not hidden); with
    ``target_column=None`` it is an inference table checked exactly as ``validate_inference_frame`` checks it.
    Rejection is reported by raising the same error the core helper raises.
    """
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (the table's id)")
    table_id = names[0] if names else "table-0"
    if target_column is None:
        if feature_columns is None:
            raise ValueError("feature_columns is required to validate an inference table")
        checked, extra = validate_inference_frame(frame, feature_columns)
        entry: dict[str, Any] = {
            "id": table_id,
            "mode": "inference",
            "rows": len(checked),
            "feature_columns": list(checked.columns),
            "extra_columns": extra,
            "missing_value_columns": {str(c): int(n) for c, n in checked.isna().sum().items() if n > 0},
        }
    else:
        clean, features, report = validate_labeled_frame(
            frame,
            target_column,
            name=table_id,
            drop_columns=drop_columns,
            min_rows=min_rows,
            require_variation=require_variation,
        )
        values = clean[target_column].to_numpy(dtype=float)
        entry = {
            "id": table_id,
            "mode": "fit",
            "rows": len(clean),
            "feature_columns": features,
            "categorical_columns": [c for c in features if not pd.api.types.is_numeric_dtype(clean[c])],
            "missing_value_columns": {
                str(c): int(n) for c, n in clean[features].isna().sum().items() if n > 0
            },
            "report": report,
            "target_summary": {
                "min": float(values.min()),
                "max": float(values.max()),
                "mean": float(values.mean()),
                "distinct": int(clean[target_column].nunique()),
            },
        }
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [entry],
        "target_column": target_column,
        "drop_columns": list(drop_columns),
        "min_rows": min_rows if target_column is not None else None,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    metrics: Mapping[str, float] | None,
    *,
    baseline: Mapping[str, float] | None = None,
    independent_test: Mapping[str, float] | None = None,
    n_holdout: int | None = None,
    n_test: int | None = None,
    target_column: str | None = None,
    selection: str | None = None,
    sample_kind: str = "sample",
    estimation: str = "single seeded split; no dispersion estimate",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    ``metrics`` / ``independent_test`` are dicts from ``regression_metrics`` (mae, rmse, r2) and ``baseline``
    from ``training_mean_baseline``; the verdict is ``sample-sanity``. Without metrics (no labelled rows) the
    verdict is ``not-measurable`` and the report says what labelled data would make the task measurable.
    """
    units = {"mae": "target units", "rmse": "target units", "r2": "unitless"}

    def _entries(source: Mapping[str, float]) -> list[dict[str, Any]]:
        unknown = sorted(set(source) - set(METRIC_IDS))
        if unknown:
            raise ValueError(f"unknown metric ids {unknown}; regression_metrics reports {list(METRIC_IDS)}")
        return [
            {
                "id": metric_id,
                "value": None if not np.isfinite(float(source[metric_id])) else float(source[metric_id]),
                "units": units[metric_id],
                "higher_is_better": metric_id == "r2",
            }
            for metric_id in METRIC_IDS
            if metric_id in source
        ]

    base: dict[str, Any] = {
        "task": "tabular regression by in-context conditioning on labelled support rows (AutoGluon Mitra)",
        "score_semantics": "continuous point predictions in target units; no per-prediction uncertainty",
        "sample_kind": sample_kind,
        "n_holdout": n_holdout,
        "n_test": n_test,
        "target_column": target_column,
        "selection": selection,
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if metrics is None:
        return {
            **base,
            "metrics": [],
            "independent_test": [],
            "verdict": "not-measurable",
            "reason": "no labelled holdout rows were supplied for the scored table",
            "needs": (
                "a labelled holdout table with a finite, varying numeric target column, scored with "
                "`regression_metrics` (mae, rmse, r2) against `training_mean_baseline`; an independent test "
                "partition from the deployment domain for any generalisable claim"
            ),
        }
    reported = [{**entry, "estimation": estimation} for entry in _entries(metrics)]
    test_entries: list[dict[str, Any]] = []
    if independent_test is not None:
        test_estimation = "independent test partition, single run"
        test_entries = [{**e, "estimation": test_estimation} for e in _entries(independent_test)]
    baselines = [] if baseline is None else [{"id": "training_mean", "metrics": _entries(baseline)}]
    rows = "an unstated number of" if n_holdout is None else str(n_holdout)
    return {
        **base,
        "metrics": reported,
        "independent_test": test_entries,
        "baselines": baselines,
        "verdict": "sample-sanity",
        "reason": f"{rows} labelled holdout row(s) from one seeded split; tutorial evidence, not a benchmark",
        "needs": (
            "an independent, domain-representative labelled test set for any generalisable quality claim; "
            "the point predictions carry no uncertainty interval"
        ),
    }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `2`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `5f277aa8f690…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `MitraRegressionPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "mitra-regressor",
  "modelId": "autogluon/mitra-regressor",
  "revision": "5f277aa8f69042d39d6ac3612aed18bb9279bd95",
  "files": [
    {
      "path": "model.safetensors",
      "bytes": 302683140,
      "sha256": "d8e75c62af0bec2fd404b0ad20a442d951d43ca6d331315cfcc0509b54f2c642"
    },
    {
      "path": "config.json",
      "bytes": 81,
      "sha256": "2bc1ed5047f7c25368245e8ad32540a5fa28940b1ec05d3f1f454a09ff5384c1"
    }
  ],
  "totalBytes": 302683221
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = MitraRegressionPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Load the sample or your own data

`Sample: Diabetes` (default) is the bundled numeric sanity check, split 60/20/20 into support, holdout and an independent test partition. `Upload CSV` takes one labelled CSV and makes a seeded random holdout (it assumes approximately IID rows); `Upload pre-split train/val/test` takes your own partitions without re-splitting — the repository's `examples/sample-data/*.zip` archives (FreshRetailNet, Insurance Charges, Ames Housing) ship exactly those three files. Columns listed in `DROP_COLUMNS` are removed before validation. Rows whose target is missing are **dropped and counted**, exact cross-partition overlaps are counted, and support rows above `MAX_TRAIN_ROWS` are capped by seeded sampling and reported. The SHA-256 of the loaded data is printed so the exported provenance can be tied to it.

In [ ]:
import hashlib

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split

DATA_SOURCE = 'Sample: Diabetes'  # @param ["Sample: Diabetes", "Upload CSV", "Upload pre-split train/val/test"]
USE_BYOD = False  # @param {type:"boolean"}
TARGET_COLUMN = 'target'  # @param {type:"string"}
DROP_COLUMNS = ''  # @param {type:"string"}
VALIDATION_SPLIT = 0.20  # @param {type:"number"}
SEED = 42  # @param {type:"integer"}
if USE_BYOD and DATA_SOURCE.startswith('Sample'):
    raise ValueError('USE_BYOD=True requires an Upload DATA_SOURCE.')
if DATA_SOURCE.startswith('Upload') and not USE_BYOD:
    raise ValueError('Set USE_BYOD=True to use an upload DATA_SOURCE.')
drop_columns = [c.strip() for c in DROP_COLUMNS.split(',') if c.strip() and c.strip() != TARGET_COLUMN]

test_data = None
if DATA_SOURCE == 'Sample: Diabetes':
    dataset = load_diabetes(as_frame=True)
    frame = dataset.frame.rename(columns={dataset.target.name: TARGET_COLUMN})
    train_data, remainder = train_test_split(frame, test_size=0.4, random_state=SEED)
    holdout_data, test_data = train_test_split(remainder, test_size=0.5, random_state=SEED)
    payloads = {'sample.csv': frame.to_csv(index=False).encode('utf-8')}
    data_name, sample_kind = 'sklearn-diabetes', 'sample'
elif DATA_SOURCE == 'Upload pre-split train/val/test':
    from google.colab import files
    uploaded = files.upload()
    by_base = {Path(name).name.lower(): payload for name, payload in uploaded.items()}
    missing = sorted({'train.csv', 'val.csv', 'test.csv'} - set(by_base))
    if missing:
        raise RuntimeError(f'Upload train.csv, val.csv, and test.csv together. Missing: {missing}')
    payloads = {name: by_base[name] for name in ('train.csv', 'val.csv', 'test.csv')}
    train_data = read_csv_bytes(payloads['train.csv'], 'train.csv')
    holdout_data = read_csv_bytes(payloads['val.csv'], 'val.csv')
    test_data = read_csv_bytes(payloads['test.csv'], 'test.csv')
    data_name, sample_kind = 'pre-split upload', 'BYOD'
else:
    from google.colab import files
    uploaded = files.upload()
    csvs = [(name, payload) for name, payload in uploaded.items() if name.lower().endswith('.csv')]
    if len(csvs) != 1:
        raise RuntimeError('Upload exactly one labelled CSV.')
    data_name, payload = csvs[0]
    payloads = {data_name: payload}
    data = read_csv_bytes(payload, data_name)
    if not 0.05 <= VALIDATION_SPLIT <= 0.40:
        raise ValueError('VALIDATION_SPLIT must be between 0.05 and 0.40.')
    train_data, holdout_data = train_test_split(data, test_size=VALIDATION_SPLIT, random_state=SEED, shuffle=True)
    sample_kind = 'BYOD'
    print('Upload CSV uses a seeded random holdout and assumes approximately IID rows.')
DATA_DIGEST = hashlib.sha256(json.dumps({name: hashlib.sha256(payload).hexdigest() for name, payload in sorted(payloads.items())}, sort_keys=True).encode()).hexdigest()
print({'sample_kind': sample_kind, 'name': data_name, 'target': TARGET_COLUMN, 'drop_columns': drop_columns, 'train_rows': len(train_data), 'holdout_rows': len(holdout_data), 'test_rows': 0 if test_data is None else len(test_data), 'data_sha256': DATA_DIGEST})

## 5. Validate the tables → input manifest, then cap and check overlaps

`validate_inputs` is the package's public validation stage: it applies exactly the checks `validate_labeled_frame` applies — unique column names, the target present and numeric, rows with a missing target dropped and counted, no infinite target, at least `MIN_TRAIN_ROWS` support rows (2 for a holdout), at most `MAX_FEATURES` features, a support target that varies — and returns an **input manifest** naming the schema, the observed structure (categorical columns, missing values, the validation report with dropped and exact-duplicate counts, a target summary) and the verdict. It is written to `outputs/mitra_regressor_input_manifest.json`. To show what rejection looks like, the cell also validates a probe with too few rows and records the package's own error message as a finding. The ceilings are printed before any model runs.

The holdout and test partitions are then re-ordered to the support schema, exact cross-partition overlaps are reported (`split_overlap_report`), support rows above `MAX_TRAIN_ROWS` are capped (`cap_training_rows`), and the trivial training-mean baseline is computed with `training_mean_baseline`. Everything in Section 6 should be read against that baseline.

In [ ]:
os.makedirs('outputs', exist_ok=True)
print({'ceilings': {'MIN_TRAIN_ROWS': MIN_TRAIN_ROWS, 'MAX_TRAIN_ROWS': MAX_TRAIN_ROWS, 'MAX_FEATURES': MAX_FEATURES}})
input_manifest = validate_inputs(train_data, target_column=TARGET_COLUMN, drop_columns=drop_columns, names=[data_name + ':train'])
input_manifest['inputs'].extend(validate_inputs(holdout_data, target_column=TARGET_COLUMN, drop_columns=drop_columns, min_rows=2, require_variation=False, names=[data_name + ':holdout'])['inputs'])
if test_data is not None:
    input_manifest['inputs'].extend(validate_inputs(test_data, target_column=TARGET_COLUMN, drop_columns=drop_columns, min_rows=2, require_variation=False, names=[data_name + ':test'])['inputs'])
# Demonstrate rejection on a probe that breaks a ceiling; the finding is recorded, not swallowed.
try:
    validate_inputs(train_data.head(MIN_TRAIN_ROWS - 1), target_column=TARGET_COLUMN, drop_columns=drop_columns)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'too-few-rows-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/mitra_regressor_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest['inputs'][0], indent=2))
print('findings:', input_manifest['findings'])

train_data, FEATURE_COLUMNS, train_report = validate_labeled_frame(train_data, TARGET_COLUMN, name='train', drop_columns=drop_columns, min_rows=MIN_TRAIN_ROWS, require_variation=True)
holdout_data, val_features, _ = validate_labeled_frame(holdout_data, TARGET_COLUMN, name='holdout', drop_columns=drop_columns)
ordered = FEATURE_COLUMNS + [TARGET_COLUMN]
if set(val_features) != set(FEATURE_COLUMNS):
    raise ValueError('train/holdout feature column names do not match.')
holdout_data = holdout_data.reindex(columns=ordered)
if test_data is not None:
    test_data, test_features, _ = validate_labeled_frame(test_data, TARGET_COLUMN, name='test', drop_columns=drop_columns)
    if set(test_features) != set(FEATURE_COLUMNS):
        raise ValueError('train/test feature column names do not match.')
    test_data = test_data.reindex(columns=ordered)
overlaps = split_overlap_report({'train': train_data[ordered], 'holdout': holdout_data, **({'test': test_data} if test_data is not None else {})})
if any(overlaps.values()):
    print('WARNING exact cross-split overlap detected; investigate leakage before interpreting metrics:', overlaps)
train_data, cap_report = cap_training_rows(train_data, TARGET_COLUMN, seed=SEED)
baseline = training_mean_baseline(train_data[TARGET_COLUMN], holdout_data[TARGET_COLUMN])
print({'train': len(train_data), 'holdout': len(holdout_data), 'test': 0 if test_data is None else len(test_data), 'features': len(FEATURE_COLUMNS), 'cap': cap_report, 'overlaps': overlaps})
if len(FEATURE_COLUMNS) > 100 or len(train_data) > 5_000:
    print('Above the <=100-feature / <=5,000-row regime where Mitra is reported to be particularly strong.')
print('training-mean baseline on the holdout', baseline)

## 6. Evaluate pretrained Mitra and executable baselines, then optionally fine-tune

`pipe.fit(...)` with `fine_tune=False` registers the support rows and the model configuration through AutoGluon (`fit_mitra_predictor`); no weight is gradient-updated. `regression_metrics` scores the holdout and, when present, the independent test partition (MAE and RMSE in target units, R² relative to a constant-mean reference); AutoGluon's own evaluation (`pipe.evaluate`) is printed alongside. Constant mean/median predictors, LightGBM and Random Forest are fitted on the exact same support rows (with train-fitted median imputation and ordinal encoding that is never refitted on holdout/test) and scored on the exact same partitions (EVAL15).

**Fine-tuning gate (off by default; GPU only).** With `RUN_FINE_TUNING=True` a second predictor is fitted with `fine_tune=True` for `FINE_TUNE_STEPS`, and it replaces the pretrained predictor **only** if it beats it on the holdout under `EVAL_METRIC` and the holdout has at least `MIN_SELECTION_HOLDOUT_ROWS` rows. The independent test is evidence only; a worse test result is surfaced as a warning and never changes the selection. **Reproducibility boundary.** `SEED` drives the split, the cap and Mitra's `random_state`; bitwise-identical results across devices and library builds are not promised.

In [ ]:
import gc
import shutil

from lightgbm import LGBMRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder

EVAL_METRIC = 'mean_absolute_error'  # @param ["mean_absolute_error", "root_mean_squared_error"]
BASELINE_TIME_LIMIT = 300  # @param {type:"integer"}
RUN_FINE_TUNING = False  # @param {type:"boolean"}
FINE_TUNE_STEPS = 50  # @param {type:"integer"}
FINE_TUNE_TIME_LIMIT = 600  # @param {type:"integer"}
MAX_MEMORY_USAGE_RATIO = 1.10  # @param {type:"number"}
MIN_SELECTION_HOLDOUT_ROWS = 50
PRETRAINED_PATH, FINETUNED_PATH = Path('outputs') / 'mitra-pretrained', Path('outputs') / 'mitra-finetuned'
for path in (PRETRAINED_PATH, FINETUNED_PATH):
    shutil.rmtree(path, ignore_errors=True)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

def score(model, frame):
    return regression_metrics(frame[TARGET_COLUMN].to_numpy(dtype=float), model.predict(frame))

pipe.fit(train_data, target_column=TARGET_COLUMN, eval_metric=EVAL_METRIC, path=PRETRAINED_PATH, fine_tune=False, time_limit=BASELINE_TIME_LIMIT, seed=SEED, max_memory_usage_ratio=MAX_MEMORY_USAGE_RATIO)
pretrained_metrics = score(pipe, holdout_data)
pretrained_test_metrics = score(pipe, test_data) if test_data is not None else None
print('pretrained holdout', pretrained_metrics, '| AutoGluon evaluate:', pipe.evaluate(holdout_data))
if pretrained_test_metrics:
    print('pretrained independent test', pretrained_test_metrics)

candidate = candidate_metrics = candidate_test_metrics = None
if RUN_FINE_TUNING:
    if not torch.cuda.is_available():
        raise RuntimeError('Fine-tuning requires a GPU. Choose Runtime -> Change runtime type -> GPU.')
    candidate = MitraRegressionPipeline(weights_path=pipe.model_weight_path, config_path=pipe.config_path, snapshot_path=pipe.snapshot_path, device=pipe.device)
    candidate.fit(train_data, target_column=TARGET_COLUMN, eval_metric=EVAL_METRIC, path=FINETUNED_PATH, fine_tune=True, fine_tune_steps=FINE_TUNE_STEPS, time_limit=FINE_TUNE_TIME_LIMIT, seed=SEED, max_memory_usage_ratio=MAX_MEMORY_USAGE_RATIO)
    candidate_metrics = score(candidate, holdout_data)
    candidate_test_metrics = score(candidate, test_data) if test_data is not None else None
    print('fine-tuned holdout', candidate_metrics)

ACTIVE_MODEL, ACTIVE_MODE, SELECTION_BASIS = pipe, 'pretrained', 'default:pretrained'
metric_key = {'mean_absolute_error': 'mae', 'root_mean_squared_error': 'rmse'}[EVAL_METRIC]
if candidate is not None:
    if len(holdout_data) < MIN_SELECTION_HOLDOUT_ROWS:
        SELECTION_BASIS = f'default:pretrained; holdout-too-small:{len(holdout_data)}<{MIN_SELECTION_HOLDOUT_ROWS}'
    else:
        SELECTION_BASIS = f'holdout:{EVAL_METRIC}'
        if candidate_metrics[metric_key] < pretrained_metrics[metric_key]:
            ACTIVE_MODEL, ACTIVE_MODE = candidate, 'fine-tuned'
    if candidate_test_metrics and pretrained_test_metrics:
        degraded = [k for k in ('mae', 'rmse') if candidate_test_metrics[k] > pretrained_test_metrics[k]] + [k for k in ('r2',) if candidate_test_metrics[k] < pretrained_test_metrics[k]]
        if degraded:
            print('WARNING independent-test metrics worsened after fine-tuning:', degraded, '(evidence only; never used for selection)')
active_metrics = candidate_metrics if ACTIVE_MODE == 'fine-tuned' else pretrained_metrics
active_test_metrics = candidate_test_metrics if ACTIVE_MODE == 'fine-tuned' else pretrained_test_metrics
print({'recommended_for_export': ACTIVE_MODE, 'selection_basis': SELECTION_BASIS, 'device': ACTIVE_MODEL.device})

# Executable baselines on the exact same partitions.
y_train = train_data[TARGET_COLUMN].to_numpy(dtype=float)
y_holdout = holdout_data[TARGET_COLUMN].to_numpy(dtype=float)
y_test = test_data[TARGET_COLUMN].to_numpy(dtype=float) if test_data is not None else None
baseline_rows = []
for strategy in ('mean', 'median'):
    dummy = DummyRegressor(strategy=strategy).fit(np.zeros((len(y_train), 1)), y_train)
    baseline_rows.append({'model': f'Dummy-{strategy}', 'split': 'holdout', **regression_metrics(y_holdout, dummy.predict(np.zeros((len(y_holdout), 1))))})
    if y_test is not None:
        baseline_rows.append({'model': f'Dummy-{strategy}', 'split': 'test', **regression_metrics(y_test, dummy.predict(np.zeros((len(y_test), 1))))})
cat_cols = [c for c in FEATURE_COLUMNS if not pd.api.types.is_numeric_dtype(train_data[c])]
num_cols = [c for c in FEATURE_COLUMNS if pd.api.types.is_numeric_dtype(train_data[c])]
num_imputer = SimpleImputer(strategy='median', keep_empty_features=True) if num_cols else None
cat_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1) if cat_cols else None

def tree_features(frame, fit=False):
    parts = []
    if num_cols:
        values = num_imputer.fit_transform(frame[num_cols]) if fit else num_imputer.transform(frame[num_cols])
        parts.append(pd.DataFrame(values, columns=num_cols, index=frame.index))
    if cat_cols:
        values = cat_encoder.fit_transform(frame[cat_cols].astype(str)) if fit else cat_encoder.transform(frame[cat_cols].astype(str))
        parts.append(pd.DataFrame(values, columns=cat_cols, index=frame.index))
    return pd.concat(parts, axis=1)[FEATURE_COLUMNS]

X_train_tree = tree_features(train_data, fit=True)
for model_name, model in {'LightGBM': LGBMRegressor(random_state=SEED, n_estimators=100, verbose=-1), 'RandomForest': RandomForestRegressor(random_state=SEED, n_estimators=100)}.items():
    model.fit(X_train_tree, y_train)
    baseline_rows.append({'model': model_name, 'split': 'holdout', **regression_metrics(y_holdout, model.predict(tree_features(holdout_data)))})
    if test_data is not None:
        baseline_rows.append({'model': model_name, 'split': 'test', **regression_metrics(y_test, model.predict(tree_features(test_data)))})
baseline_rows.append({'model': f'Mitra-{ACTIVE_MODE}', 'split': 'holdout', **active_metrics})
if active_test_metrics:
    baseline_rows.append({'model': f'Mitra-{ACTIVE_MODE}', 'split': 'test', **active_test_metrics})
metrics_table = pd.DataFrame(baseline_rows)
print(metrics_table.to_string(index=False))
print('All values above are current-run tutorial metrics. Lower MAE/RMSE is better; higher R2 is better.')

## 7. Evaluate → evaluation report

`evaluation_report` is the package's public evaluation stage and always produces a report. Here it carries the active model's holdout metrics (`mae`, `rmse`, `r2` — the repository's own metric ids), the independent-test metrics when a test partition exists, and the training-mean baseline, with the verdict `sample-sanity`: one seeded split with no dispersion estimate, tutorial evidence rather than a benchmark; the executable-baseline table is attached. Without a labelled holdout the verdict would be `not-measurable`. The report is written to `outputs/mitra_regressor_evaluation_report.json`.

In [ ]:
report = evaluation_report(active_metrics, baseline=baseline, independent_test=active_test_metrics, n_holdout=len(holdout_data), n_test=None if test_data is None else len(test_data), target_column=TARGET_COLUMN, selection=SELECTION_BASIS, sample_kind=sample_kind, estimation='single seeded split (support/holdout/independent test); no dispersion estimate')
report['executable_baselines'] = baseline_rows
with open('outputs/mitra_regressor_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps({key: report[key] for key in ('verdict', 'reason', 'selection', 'n_holdout', 'n_test')}, indent=2))

## 8. Optional new-data point prediction

Off by default so a top-to-bottom run needs no upload dialog. Switch `RUN_NEW_DATA_INFERENCE` on and upload one CSV with the support feature columns (extra columns are preserved in the output and not passed to the model), or set `NEW_DATA_PATH` for a non-interactive executor. `validate_inputs(..., target_column=None, feature_columns=...)` applies exactly the checks `validate_inference_frame` applies — unique header, every feature present, no pre-existing `prediction` column. Predictions are **continuous point estimates only**; on the sample path eight held-out rows are scored instead so a prediction CSV always exists.

In [ ]:
RUN_NEW_DATA_INFERENCE = False  # @param {type:"boolean"}
NEW_DATA_PATH = ''  # @param {type:"string"}
new_data_result = None
if RUN_NEW_DATA_INFERENCE:
    if NEW_DATA_PATH:
        csv_name, payload = os.path.basename(NEW_DATA_PATH), Path(NEW_DATA_PATH).read_bytes()
    else:
        from google.colab import files
        new_upload = files.upload()
        csvs = [(name, payload) for name, payload in new_upload.items() if name.lower().endswith('.csv')]
        if len(csvs) != 1:
            raise RuntimeError('Upload exactly one inference CSV.')
        csv_name, payload = csvs[0]
    new_data = read_csv_bytes(payload, csv_name)
    inference_manifest = validate_inputs(new_data, None, feature_columns=FEATURE_COLUMNS, names=[csv_name])
    X_new, extra_columns = validate_inference_frame(new_data, FEATURE_COLUMNS)
    out = new_data.copy()
    out['prediction'] = ACTIVE_MODEL.predict(X_new)
    new_data_result = {'input': csv_name, 'rows': len(out), 'extra_columns': extra_columns, 'input_manifest': inference_manifest}
else:
    out = holdout_data[FEATURE_COLUMNS].head(8).copy()
    out['prediction'] = ACTIVE_MODEL.predict(out)
    print('Inference upload skipped; eight held-out rows scored instead.')
out.to_csv('outputs/mitra_regressor_predictions.csv', index=False)
print(out.head())

## 9. Export the deployable predictor bundle, then prove a fresh reload

The deployable artifact is the selected AutoGluon predictor directory (ART1), not a replacement `model.safetensors`: for Mitra it contains the registered support rows, the model configuration and AutoGluon's serialised state, so it inherits the source data's confidentiality, licensing, retention and disclosure obligations (ART3/ART7). `tutorial_run_metadata.json` records the base-model identity and digests, the AutoGluon/Python versions, the target and features, the mode and selection basis, the data digest and the metrics; `write_artifact_manifest` inventories every file with its size and SHA-256 (ART5/ART6). The directory is zipped and its SHA-256 printed.

An in-memory predictor is not evidence that serialisation worked. The cell extracts the exact ZIP into a fresh directory with `safe_extract_archive` (path, symlink, size and compression-ratio checks; never `extractall`), verifies the manifest and provenance with `validate_artifact_directory` **before** deserialising, reloads the predictor and checks that its predictions agree with the in-memory model's on eight held-out rows within `rtol=1e-6, atol=1e-8` (VER1–VER5). The result JSON then records everything: predictions, metrics, the evaluation report, the input manifest, the data digest, the bundle identity, the notebook's source, the model identity, revision and licence, and the runtime.

In [ ]:
from datetime import datetime, timezone

from autogluon.tabular import TabularPredictor

active_path = Path(ACTIVE_MODEL.predictor.path)
run_metadata = {
    'artifact_format': ARTIFACT_FORMAT, 'artifact_format_version': ARTIFACT_FORMAT_VERSION,
    'base_model': MODEL_ID, 'base_model_revision': MODEL_REVISION, 'weights_sha256': WEIGHTS_SHA256, 'config_sha256': CONFIG_SHA256,
    'notebook_source': NOTEBOOK_SOURCE, 'model_source': 'verified local snapshot (Section 3)',
    'autogluon_version': importlib.metadata.version('autogluon.tabular'), 'python_version': platform.python_version(), 'torch_version': torch.__version__, 'device': ACTIVE_MODEL.device,
    'problem_type': 'regression', 'target_column': TARGET_COLUMN, 'features': FEATURE_COLUMNS,
    'mode': ACTIVE_MODE, 'selection_basis': SELECTION_BASIS, 'seed': SEED, 'data_source': DATA_SOURCE, 'data_sha256': DATA_DIGEST,
    'train_rows_before_cap': cap_report['before'], 'train_rows_used': len(train_data), 'train_row_cap_applied': cap_report['applied'],
    'holdout_rows': len(holdout_data), 'independent_test_rows': None if test_data is None else len(test_data), 'eval_metric': EVAL_METRIC,
    'fine_tuning_requested': RUN_FINE_TUNING, 'fine_tune_steps_requested': FINE_TUNE_STEPS if RUN_FINE_TUNING else None, 'fine_tune_time_limit_seconds': FINE_TUNE_TIME_LIMIT if RUN_FINE_TUNING else None, 'max_memory_usage_ratio': MAX_MEMORY_USAGE_RATIO,
    'pretrained_holdout_metrics': pretrained_metrics, 'pretrained_test_metrics': pretrained_test_metrics, 'finetuned_holdout_metrics': candidate_metrics, 'finetuned_test_metrics': candidate_test_metrics,
    'prediction_uncertainty': 'point predictions only; no calibrated per-prediction interval', 'exported_at_utc': datetime.now(timezone.utc).isoformat(),
}
(active_path / 'tutorial_run_metadata.json').write_text(json.dumps(run_metadata, indent=2), encoding='utf-8')
manifest_path = write_artifact_manifest(active_path)
archive_base = Path('outputs') / 'mitra_regressor_predictor'
Path(str(archive_base) + '.zip').unlink(missing_ok=True)
archive = Path(shutil.make_archive(str(archive_base), 'zip', root_dir=active_path))
archive_digest = sha256_file(archive)
print({'predictor_zip': str(archive), 'zip_sha256': archive_digest, 'artifact_manifest': str(manifest_path)})

RELOAD_DIR = Path('outputs') / 'artifact-reload'
shutil.rmtree(RELOAD_DIR, ignore_errors=True)
safe_extract_archive(archive, RELOAD_DIR)
verified_manifest, verified_metadata = validate_artifact_directory(RELOAD_DIR)
if verified_metadata['autogluon_version'] != importlib.metadata.version('autogluon.tabular'):
    raise RuntimeError('Artifact/runtime AutoGluon version mismatch.')
reloaded = TabularPredictor.load(str(RELOAD_DIR))
smoke_X = holdout_data[FEATURE_COLUMNS].head(8).copy()
np.testing.assert_allclose(ACTIVE_MODEL.predict(smoke_X), predict_regression(reloaded, smoke_X, FEATURE_COLUMNS), rtol=1e-6, atol=1e-8)
print('PASS: artifact manifest/provenance verified before deserialisation; predictor reloaded from fresh files; predictions equivalent (rtol=1e-6, atol=1e-8).')

payload = {
    'predictions': out.to_dict(orient='records'),
    'new_data': new_data_result,
    'metrics': {'active_mode': ACTIVE_MODE, 'holdout': active_metrics, 'independent_test': active_test_metrics, 'pretrained_holdout': pretrained_metrics, 'finetuned_holdout': candidate_metrics, 'executable_baselines': baseline_rows},
    'training_mean_baseline': baseline,
    'evaluation_report': report,
    'input_manifest': input_manifest,
    'sample': {'kind': sample_kind, 'name': data_name, 'source': DATA_SOURCE, 'data_sha256': DATA_DIGEST, 'train_rows': len(train_data), 'holdout_rows': len(holdout_data), 'test_rows': 0 if test_data is None else len(test_data)},
    'artifact': {'zip': archive.name, 'zip_sha256': archive_digest, 'run_metadata': run_metadata},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'autogluon': importlib.metadata.version('autogluon.tabular'), 'lightgbm': importlib.metadata.version('lightgbm'), 'numpy': numpy.__version__, 'pandas': pandas.__version__, 'sklearn': sklearn.__version__, 'device': ACTIVE_MODEL.device},
}
with open('outputs/mitra_regressor_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

Predictions are continuous point estimates in the target's units with no uncertainty interval; any tolerance band must be chosen on the caller's own labelled, domain-representative data. The evaluation report's `sample-sanity` verdict names what it is: one seeded split of a public sample with no dispersion estimate — tutorial evidence that must not be generalised. The executable baselines show when the foundation model adds value on this table; the fine-tuned variant, when requested, is selected on the holdout only. Rows that are not independent, targets outside the support range, cross-partition overlaps, tables above the ≤100-feature / ≤5,000-row regime and capped support sets all change results in ways these metrics do not measure.

Successful execution proves that the recorded repository revision's package, carried in this notebook, can acquire and digest-verify the pinned checkpoint and stage it offline, validate the demonstrated tables, register regression support rows through AutoGluon, compute sample metrics against trivial and classical baselines, write the input manifest and the evaluation report, export the deployable predictor bundle with its manifest and provenance, and reload an equivalent predictor from that bundle alone — without the repository being reachable. It does **not** establish benchmark superiority, domain generalisation, fairness, robustness, calibration, production safety, or deployment fitness.

**Next experiments:** switch `DATA_SOURCE` to `Upload pre-split train/val/test` with one of the repository's `examples/sample-data` archives (FreshRetailNet, Insurance Charges, Ames Housing) or your own partitions; enable `RUN_FINE_TUNING` on a GPU runtime and watch the holdout-based selection and the independent-test warning; feed the exported `outputs/mitra_regressor_predictor.zip` to the companion predictor-inference notebook in a separate session.

## References

- Repository README: https://github.com/kurtvalcorza/mitra-regressor-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/mitra-regressor-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/mitra-regressor-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/autogluon/mitra-regressor
- Upstream library: https://github.com/autogluon/autogluon
- Mitra paper: https://arxiv.org/abs/2508.02927